In [ ]:
# !pip install -q -U transformers torch huggingface_hub accelerate optimum bitsandbytes sentencepiece tqdm pandas
# !pip install -q -U auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/ # Adjust cu118 if needed

In [ ]:
# @title 1. Installs and Imports
# Install necessary libraries (run this cell first)


import numpy as np
import pandas as pd
import os
import torch
import gc
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from auto_gptq import AutoGPTQForCausalLM
from huggingface_hub import login

# Register the tqdm pandas extension for progress bars with DataFrame.apply
tqdm.pandas()

print("Libraries installed and imported.")

/usr/local/lib/python3.11/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.11/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:418: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.11/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float16)


Libraries installed and imported.


In [ ]:
# @title 2. Configuration and Hugging Face Login

# --- Configuration Settings ---
HUGGING_FACE_TOKEN = "<INSERT_YOUR_HUGGING_FACE_TOKEN>" # Optional: Set your Hugging Face token here

# Model identifier on Hugging Face Hub
MODEL_NAME = "TheBloke/sqlcoder-7B-GPTQ"
# MODEL_NAME = "defog/sqlcoder-7b-2"    Alternative model if AutoGPTQ fails or for different quantization

DATA_PATH = "/dataset/train.csv" # Make sure this file exists if you uncomment DatasetProcessor usage

# --- Generation Parameters ---
MAX_NEW_TOKENS = 512 # Max tokens for the generated SQL query
TEMPERATURE = 0.1    # Generation temperature (lower values are more deterministic)

# --- Hardware Settings ---
USE_GPU = torch.cuda.is_available()
DEVICE = "cuda:0" if USE_GPU else "cpu"
print(f"Using device: {DEVICE}")

# --- Login to Hugging Face ---
if HUGGING_FACE_TOKEN:
    print("Logging into Hugging Face Hub...")
    try:
        login(token=HUGGING_FACE_TOKEN)
        print("Login successful.")
    except Exception as e:
        print(f"Login failed: {e}")
else:
    print("Warning: Hugging Face token not provided. Downloads might fail for gated models.")

Using device: cuda:0
Logging into Hugging Face Hub...
Login successful.


In [ ]:
# @title 3. Helper Function: Memory Management

def flush_memory():
    """Clears GPU cache and runs Python's garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # torch.cuda.reset_peak_memory_stats() # Uncomment for detailed memory debugging

print("Memory flushing function defined.")

Memory flushing function defined.


In [ ]:
# @title 4. Core Function: Load Quantized Model

def load_quantized_model(model_name, use_gpu=True):
    """
    Loads the specified model and tokenizer.
    Attempts AutoGPTQ first, then falls back to BitsAndBytes 4-bit,
    then attempts loading without quantization.
    """
    print(f"Loading model: {model_name}...")
    device = "cuda:0" if torch.cuda.is_available() and use_gpu else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    print(f"Tokenizer loaded for {model_name}.")

    # Add pad token if missing (common for Llama-based models)
    if tokenizer.pad_token is None:
        print("Adding padding token '[PAD]' to tokenizer.")
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})

    model = None
    # --- Attempt 1: AutoGPTQ ---
    try:
        print("Attempting to load model with AutoGPTQ...")
        model = AutoGPTQForCausalLM.from_quantized(
            model_name,
            use_safetensors=True,
            trust_remote_code=True, # Caution: Only use if you trust the source
            device=device,          # Load directly to the target device
            use_triton=False,
            quantize_config=None,
            inject_fused_attention=False,
            inject_fused_mlp=False,
        )
        # Resize embeddings if pad token was added
        if tokenizer.pad_token == '[PAD]':
             model.resize_token_embeddings(len(tokenizer))
        print("AutoGPTQ model loaded successfully.")

    except Exception as e_gptq:
        print(f"AutoGPTQ loading failed: {e_gptq}")
        # --- Attempt 2: BitsAndBytes 4-bit ---
        print("Attempting to load with BitsAndBytes quantization (4-bit)...")
        try:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16, # Computation type
                bnb_4bit_quant_type="nf4",           # Quantization type (nf4 recommended)
                bnb_4bit_use_double_quant=True,      # Use double quantization for potential memory savings
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=quantization_config,
                torch_dtype=torch.float16, # Use float16 for inference
                device_map="auto",         # Let accelerate handle device placement
                trust_remote_code=True,    # Caution needed
            )
            if tokenizer.pad_token == '[PAD]':
                 model.resize_token_embeddings(len(tokenizer))
            print("BitsAndBytes 4-bit model loaded successfully.")

        except Exception as e_bnb:
            print(f"BitsAndBytes loading also failed: {e_bnb}")
            # --- Attempt 3: No Quantization (High Resource Usage) ---
            print("Attempting to load without quantization...")
            try:
                 model = AutoModelForCausalLM.from_pretrained(
                     model_name,
                     torch_dtype=torch.float16,
                     device_map="auto",
                     trust_remote_code=True,
                 )
                 if tokenizer.pad_token == '[PAD]':
                     model.resize_token_embeddings(len(tokenizer))
                 print("Non-quantized model loaded (may use significant resources).")
            except Exception as e_plain:
                 print(f"ERROR: Failed to load model in any configuration: {e_plain}")
                 return None, None # Return None if all attempts fail

    # Ensure model is on the correct primary device if device_map="auto" was used implicitly
    # model.to(device) # Often handled by device_map but can be added for explicit control if needed

    return tokenizer, model

print("Model loading function defined.")

Model loading function defined.


In [ ]:
# @title 5. Core Functions: Prompting, Generation, Extraction

def create_prompt(question, schema):
    """Creates a structured prompt for the SQL generation task."""
    system_prompt = """### Instructions:
Given the database schema and a question, generate the SQL query that answers the question.
Respond ONLY with the SQL query. Start your response with the marker 'Sql Query:'. Do not include explanations or markdown formatting like ```sql.

### Schema:
{schema}

### Question:
{question}

### Response:
Sql Query:"""
    # Use .format() for safer substitution than f-strings in this context
    return system_prompt.format(schema=schema, question=question)


def generate_sql_query(tokenizer, model, prompt, device, max_new_tokens=512, temperature=0.1):
    """Generates the SQL query using the loaded model and tokenizer."""
    flush_memory() # Clear memory before generation

    inputs = tokenizer(prompt, return_tensors="pt", padding=False, truncation=True).to(device)

    # --- Generation Configuration ---
    generation_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "max_new_tokens": max_new_tokens,          # Max tokens to generate for the answer
        "temperature": temperature if temperature > 0 else None, # Control randomness (0 = deterministic greedy)
        "do_sample": temperature > 0,              # Required for temperature > 0
        "num_return_sequences": 1,                 # Generate one response
        "pad_token_id": tokenizer.eos_token_id,    # Use EOS token for padding during generation
        "eos_token_id": tokenizer.eos_token_id     # Stop generation at EOS token
    }

    generated_text = "Error: Generation failed" # Default error message
    try:
        with torch.no_grad(): # Disable gradient calculation for inference
            outputs = model.generate(**generation_kwargs)

        # Decode the generated token IDs back to text, skipping special tokens (like padding/EOS)
        # outputs[0] takes the first sequence if num_return_sequences > 1
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # We return the full response including the prompt for easier extraction later
        generated_text = full_response

    except Exception as e:
        print(f"Error during generation: {e}")
        # Include the error in the returned text for debugging
        generated_text = f"{prompt}Error generating SQL: {e}"
    finally:
        flush_memory() # Clean up memory after generation attempt

    return generated_text


def extract_sql_query(full_text):
    """
    Extracts the SQL query from the model's full output text,
    searching for the 'Sql Query:' marker (case-insensitive).
    Includes fallbacks for common failure modes.
    """
    marker = "Sql Query:"
    response_marker = "### Response:" # Used as a fallback separator

    # Find the position of the marker, case-insensitive
    marker_pos = full_text.lower().find(marker.lower())

    if marker_pos != -1:
        # Found the marker, extract text after it
        actual_start = marker_pos + len(marker)
        sql_query = full_text[actual_start:].strip()

        # Clean up potential markdown backticks if the model included them
        if sql_query.startswith("```sql"):
            sql_query = sql_query[len("```sql"):].strip()
        if sql_query.endswith("```"):
            sql_query = sql_query[:-len("```")].strip()
        return sql_query

    elif "Error generating SQL:" in full_text:
        # If the generation function caught an error, extract that message
        try:
            return full_text.split(response_marker)[-1].strip()
        except: return "Error generating SQL (extraction failed)." # Fallback

    else:
        # Fallback: Marker not found. Try extracting everything after "### Response:"
        response_pos = full_text.lower().find(response_marker.lower())
        if response_pos != -1:
            potential_query = full_text[response_pos + len(response_marker):].strip()
            # Basic check if it resembles SQL
            if potential_query.upper().startswith(("SELECT", "WITH", "INSERT", "UPDATE", "DELETE", "Error:")):
                 print(f"Warning: '{marker}' not found. Assuming text after '{response_marker}' is the query/response.")
                 return potential_query
            else:
                 # Could be explanation text if model ignored instructions
                 print(f"Warning: '{marker}' not found and text after '{response_marker}' doesn't look like SQL.")
                 return f"Query marker not found. Response: {potential_query[:200]}..." # Return beginning of response
        else: # Neither marker found
             print(f"Warning: Neither '{marker}' nor '{response_marker}' found in text: '{full_text[:200]}...'")
             return "Query marker not found."

print("Prompt creation, generation, and extraction functions defined.")

Prompt creation, generation, and extraction functions defined.


In [ ]:
# @title 6. [Optional] Dataset Processing Class (for batch processing CSV)

class DatasetProcessor:
    """Handles loading data from CSV and running batch generation."""
    def __init__(self, path, model, tokenizer, device, sample_size=None):
        self.path = path
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.sample_size = sample_size
        self.df = self._load_data() # Load data on initialization

    def _load_data(self):
        """Loads DataFrame from CSV path."""
        print(f"Loading data from: {self.path}")
        try:
            df = pd.read_csv(self.path)
            if self.sample_size:
                print(f"Taking a sample of {self.sample_size} rows.")
                df = df.head(self.sample_size)
            df["generated_sql_query"] = None # Initialize output column
            print(f"Data loaded successfully. Shape: {df.shape}")
            return df
        except FileNotFoundError:
            print(f"Error: Data file not found at {self.path}")
            return None
        except Exception as e:
            print(f"Error loading data: {e}")
            return None

    def _process_row(self, natural_language, schema):
        """Generates and extracts SQL for a single DataFrame row."""
        if pd.isna(natural_language) or pd.isna(schema):
            return "Error: Missing question or schema"

        prompt = create_prompt(natural_language, schema)
        try:
            # Call the main generation function
            generated_text = generate_sql_query(
                self.tokenizer,
                self.model,
                prompt,
                device=self.device,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE
            )
            # Extract the SQL part from the full response
            sql_query = extract_sql_query(generated_text)
            return sql_query
        except Exception as e:
            print(f"Error processing question '{str(natural_language)[:50]}...': {e}")
            # Return error message in the DataFrame column
            return f"Error during processing: {e}"

    def generate_queries(self):
        """Applies the generation process row-by-row to the loaded DataFrame."""
        if self.df is None:
            print("DataFrame not loaded. Cannot generate queries.")
            return None

        print(f"Starting SQL query generation for {len(self.df)} rows...")
        # Use pandas apply with tqdm progress bar (requires tqdm.pandas() to be run)
        # Assumes 'question' and 'schema' columns exist in the CSV
        self.df["generated_sql_query"] = self.df.progress_apply(
            lambda row: self._process_row(row["question"], row["schema"]),
            axis=1
        )

        # # --- Alternative: Manual loop if progress_apply fails ---
        # results = []
        # for index, row in tqdm(self.df.iterrows(), total=self.df.shape[0], desc="Generating SQL"):
        #   results.append(self._process_row(row["question"], row["schema"]))
        # self.df["generated_sql_query"] = results # Assign the results list back
        # # --- End Alternative ---

        print("SQL query generation complete.")
        return self.df

print("DatasetProcessor class defined (for batch processing).")

DatasetProcessor class defined (for batch processing).


In [ ]:
# @title 7. Main Execution: Interactive Mode

# --- Load the Model and Tokenizer ---
# This assumes Cells 1-5 have been run successfully
tokenizer, model = load_quantized_model(MODEL_NAME, use_gpu=USE_GPU)

# Proceed only if model loading was successful
if tokenizer and model:

    # --- Define Your Database Schema ---
    # IMPORTANT: Replace the example schema below with the actual CREATE TABLE
    # statements for YOUR database. The model NEEDS this context.
    DATABASE_SCHEMA = """
CREATE TABLE Employees (
    EmployeeID INT PRIMARY KEY,
    FirstName VARCHAR(50),
    LastName VARCHAR(50),
    DepartmentID INT,
    Salary DECIMAL(10, 2),
    HireDate DATE
);

CREATE TABLE Departments (
    DepartmentID INT PRIMARY KEY,
    DepartmentName VARCHAR(50)
);

-- Example: Add foreign key constraint if applicable (helps model understand joins)
-- ALTER TABLE Employees ADD CONSTRAINT FK_Department FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID);

/*
Tips for defining schema:
- Use standard SQL CREATE TABLE syntax.
- Include all relevant tables for your potential questions.
- Include column names and their data types (VARCHAR, INT, DATE, DECIMAL, etc.).
- Specify PRIMARY KEYs.
- Specify FOREIGN KEYs to show relationships between tables.
- Keep it concise but complete for the scope of your questions.
- Comments (like this one) within the schema string are ignored by the model.
*/
"""
    print("--- Database Schema Defined (Review and Modify in Code Above) ---")
    print(DATABASE_SCHEMA)
    print("-" * 50)


    # --- Interactive Loop ---
    print("\n--- Interactive SQL Query Mode ---")
    print("Type your natural language question and press Enter.")
    print("Type 'quit' or 'exit' to stop.")
    print("-" * 50)

    while True:
        try:
            # Get user input
            user_question = input("Your Question: ")
            user_question = user_question.strip() # Remove leading/trailing whitespace

            # Check if input is empty
            if not user_question:
                continue

            # Check for exit command
            if user_question.lower() in ['quit', 'exit']:
                print("Exiting interactive mode.")
                break

            # --- Generate SQL ---
            print("Generating SQL...")
            # 1. Create the prompt using the predefined schema and user question
            prompt = create_prompt(user_question, DATABASE_SCHEMA)

            # 2. Generate the full text response using the model
            generated_text = generate_sql_query(
                tokenizer,
                model,
                prompt,
                device=DEVICE,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE
            )

            # 3. Extract the SQL part from the model's potentially longer response
            sql_query = extract_sql_query(generated_text)

            # --- Display Result ---
            print("-" * 30)
            print(f"Generated SQL:\n{sql_query}")
            print("-" * 30 + "\n")

            # Optional: More aggressive memory clearing if needed between queries
            # flush_memory()

        except KeyboardInterrupt:
            # Handle Ctrl+C gracefully
             print("\nExiting interactive mode due to KeyboardInterrupt.")
             break
        except Exception as e:
            # Catch any other unexpected errors during the loop
            print(f"\nAn unexpected error occurred: {e}")
            print("Please try again or type 'quit'/'exit' to stop.")
            # Optional: flush memory after an error
            # flush_memory()

else:
    # This message appears if tokenizer or model is None after attempting to load
    print("Model loading failed in Cell 4. Cannot proceed to interactive mode.")

# --- End of Interactive Script ---
print("Script finished.")

Loading model: TheBloke/sqlcoder-7B-GPTQ...
Tokenizer loaded for TheBloke/sqlcoder-7B-GPTQ.
Adding padding token '[PAD]' to tokenizer.
Attempting to load model with AutoGPTQ...


WARNING - Exllamav2 kernel is not installed, reset disable_exllamav2 to True. This may because you installed auto_gptq using a pre-build wheel on Windows, in which exllama_kernels are not compiled. To use exllama_kernels to further speedup inference, you can re-install auto_gptq from source.
WARNING - CUDA kernels for auto_gptq are not installed, this will result in very slow inference speed. This may because:
1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.
1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.
INFO - The layer lm_head is not quantized.
INFO:auto_gptq.modeling._base:The layer lm_head is not quantized.


AutoGPTQ model loaded successfully.
--- Database Schema Defined (Review and Modify in Code Above) ---

CREATE TABLE Employees (
    EmployeeID INT PRIMARY KEY,
    FirstName VARCHAR(50),
    LastName VARCHAR(50),
    DepartmentID INT,
    Salary DECIMAL(10, 2),
    HireDate DATE
);

CREATE TABLE Departments (
    DepartmentID INT PRIMARY KEY,
    DepartmentName VARCHAR(50)
);

-- Example: Add foreign key constraint if applicable (helps model understand joins)
-- ALTER TABLE Employees ADD CONSTRAINT FK_Department FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID);

/*
Tips for defining schema:
- Use standard SQL CREATE TABLE syntax.
- Include all relevant tables for your potential questions.
- Include column names and their data types (VARCHAR, INT, DATE, DECIMAL, etc.).
- Specify PRIMARY KEYs.
- Specify FOREIGN KEYs to show relationships between tables.
- Keep it concise but complete for the scope of your questions.
- Comments (like this one) within the schema string are i

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Generating SQL...
Error during generation: Unexpected floating ScalarType in at::autocast::prioritize
------------------------------
Generated SQL:
'. Do not include explanations or markdown formatting like ```sql.

### Schema:

CREATE TABLE Employees (
    EmployeeID INT PRIMARY KEY,
    FirstName VARCHAR(50),
    LastName VARCHAR(50),
    DepartmentID INT,
    Salary DECIMAL(10, 2),
    HireDate DATE
);

CREATE TABLE Departments (
    DepartmentID INT PRIMARY KEY,
    DepartmentName VARCHAR(50)
);

-- Example: Add foreign key constraint if applicable (helps model understand joins)
-- ALTER TABLE Employees ADD CONSTRAINT FK_Department FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID);

/*
Tips for defining schema:
- Use standard SQL CREATE TABLE syntax.
- Include all relevant tables for your potential questions.
- Include column names and their data types (VARCHAR, INT, DATE, DECIMAL, etc.).
- Specify PRIMARY KEYs.
- Specify FOREIGN KEYs to show relationships between tab